In [ ]:
from adaptive_reward_checkpoint_fresh import run_adaptive_reward_from_snapshot

checkpoint_path = r"output_adaptive_reward_snapshot_build_SPY_252days_at_2020\year_start_checkpoints\SPY_adaptive_reward_fresh_start_252days_at_2020__continued__start_2020.joblib"

res = run_adaptive_reward_from_snapshot(
    snapshot_path=checkpoint_path,

    # Simulate through this local-data timestamp.
    end_time="2026-06-12 16:00:00",

    # Actual trading begins here.
    # The first trading day will use the latest prior p_day/gate from the checkpoint/history.
    trade_start="2020-01-01 09:30:00",

    # Optional; defaults to checkpoint_time + 1 day.
    # Keep this at or before trade_start if you want all trade_start signals considered.
    sim_start="2020-01-01 09:30:00",

    output_dir=r"output_prev_checkpoint_local_SPY_20200101_20260612",
    save_snapshot_path=r"output_prev_checkpoint_local_SPY_20200101_20260612\continued_checkpoint.joblib",

    # Start clean/flat from this capital.
    reset_execution_state=True,
    initial_capital=100000.0,
    fee_pct=0.0,

    # Keep saved model/checkpoint logic.
    autosave_year_start_checkpoints=False,
    verbose=True,
)

print("Output dir:", res["output_dir"])
print("Continued checkpoint:", res["continued_snapshot_path"])
print(res["daily_reward_df"].tail())
print(res["signal_decisions_all_df"].tail())

In [ ]:
from adaptive_reward_checkpoint_fresh import (
    build_adaptive_reward_snapshot,
    run_adaptive_reward_from_snapshot_chronological,
    load_joblib,
)
from adaptive_trade_extensions import make_ret_grid

old_checkpoint = r"output_adaptive_reward_snapshot_build_TQQQ_252days_at_2020\year_start_checkpoints\final_checkpoint__start_2020.joblib"
old = load_joblib(old_checkpoint)

build_dir = r"output_TQQQ_build_direct_gate_open_close_reward_at_2020"
checkpoint = build_dir + r"\direct_gate_checkpoint_before_2020.joblib"

build_res = build_adaptive_reward_snapshot(
    daily_csv_path=old["daily_csv_path"],
    k5m_csv_path=old["k5m_csv_path"],
    snapshot_path=checkpoint,

    code=old.get("code", "TQQQ"),
    daily_chan_start=old["original_daily_chan_start"],
    accumulation_start=old["original_accumulation_start"],
    snapshot_end_time="2019-12-31 16:00:00",

    threshold_ret_grid=make_ret_grid(5, 25, 0.5),

    daily_gate_mode="direct",
    daily_reward_mode="open_close_forced",
    daily_direct_gate_min_samples=60,

    output_dir=build_dir,
    autosave_year_start_checkpoints=False,
    verbose=True,
)

res = run_adaptive_reward_from_snapshot_chronological(
    snapshot_path=checkpoint,
    end_time="2026-06-12 16:00:00",
    sim_start="2020-01-01 09:30:00",
    trade_start="2020-01-01 09:30:00",

    output_dir=r"output_TQQQ_direct_gate_open_close_reward_2020_20260612",
    save_snapshot_path=r"output_TQQQ_direct_gate_open_close_reward_2020_20260612\continued_checkpoint.joblib",

    reset_execution_state=True,
    initial_capital=100000.0,
    fee_pct=0.0,

    threshold_ret_grid_override=make_ret_grid(5, 25, 0.5),

    daily_gate_mode="direct",
    daily_reward_mode="open_close_forced",
    daily_direct_gate_min_samples=60,

    autosave_year_start_checkpoints=False,
    verbose=True,
)

print("Output dir:", res["output_dir"])
print("Continued checkpoint:", res["continued_snapshot_path"])

print(res["daily_log_df"][[
    "date",
    "daily_action",
    "daily_gate_mode",
    "daily_reward_mode",
    "p_day",
    "direct_gate_prob_force_buy",
    "direct_gate_prob_free",
    "direct_gate_prob_force_sell",
]].tail(30))

print(res["daily_reward_df"].groupby("chosen_action")["chosen_reward"].agg(
    ["count", "mean", "median", "sum", "std", "min", "max"]
))

In [1]:
from pathlib import Path
import pandas as pd

from adaptive_reward_checkpoint_fresh import (
    build_adaptive_reward_snapshot,
    run_adaptive_reward_from_snapshot_chronological,
)

from adaptive_trade_extensions import (
    RollingThresholdConfig,
    make_threshold_grid,
    make_ret_grid,
)

five_min_grid = make_ret_grid(0, 25, 0.5)

common_kwargs = dict(
    daily_csv_path="DataAPI/data/SPY_day.csv",
    k5m_csv_path="DataAPI/data/SPY_5M.csv",
    code="SPY",
    daily_chan_start="2010-02-11",
    accumulation_start="2012-02-11",
    N_confirm=5,
    min_labeled_days_to_train=200,
    retrain_every_new_labels=25,
    dp_lookback=252,
    lookahead_days_5m=2.0,
    retrain_every_days_5m=5,
    min_samples_total_5m=300,
    threshold_window_days=2.0,
    threshold_ret_grid=five_min_grid,
    threshold_min_open_signals=10,
    initial_capital=100000.0,
    fee_pct=0.0,
    daily_chan_max_klines=500,
    five_chan_max_klines=500,
    macro_files={"vix_": "VIX.csv"},
    static_buy_level=0.20,
    static_sell_level=0.30,
    daily_threshold_config=RollingThresholdConfig(
        lookback_days=252,
        buy_grid=make_threshold_grid(0.05, 0.35, 0.005),
        sell_grid=make_threshold_grid(0.15, 0.60, 0.005),
        min_gap=0.02,
        min_obs=60,
        switch_penalty=0.0,
    ),
    verbose=True,
)

build_dir = Path("output_SPY_build_prev_daily_context_at_2020")
checkpoint_path = build_dir / "fresh_checkpoint_prev_daily_context__before_2020.joblib"

# Build the clean pre-2020 checkpoint only if it does not already exist.
if checkpoint_path.exists():
    print("Using existing clean checkpoint:", checkpoint_path)
else:
    build_res = build_adaptive_reward_snapshot(
        snapshot_path=str(checkpoint_path),
        snapshot_end_time="2019-12-31 16:00:00",
        output_dir=str(build_dir),
        autosave_year_start_checkpoints=False,
        **common_kwargs,
    )

    print("Built checkpoint:", build_res["snapshot_path"])
    display(build_res["daily_reward_df"].tail())
    display(build_res["daily_log_df"].tail())

base_output_dir = Path("output_SPY_chrono_prev_daily_context_wide_5m_resumable")
base_output_dir.mkdir(parents=True, exist_ok=True)

chunks = [
    ("2020", "2020-12-31 16:00:00"),
    ("2021", "2021-12-31 16:00:00"),
    ("2022", "2022-12-31 16:00:00"),
    ("2023", "2023-12-31 16:00:00"),
    ("2024", "2024-12-31 16:00:00"),
    ("2025", "2025-12-31 16:00:00"),
    ("2026", "2026-06-12 16:00:00"),
]

current_checkpoint = checkpoint_path
last_res = None

for year, chunk_end_time in chunks:
    chunk_dir = base_output_dir / f"chunk_{year}"
    chunk_checkpoint = chunk_dir / f"continued_checkpoint__through_{year}.joblib"

    # If this chunk already finished in a previous run, skip it and resume from it.
    if chunk_checkpoint.exists():
        print(f"[SKIP] {year} already completed:", chunk_checkpoint)
        current_checkpoint = chunk_checkpoint
        continue

    print(f"[RUN] {year}: {current_checkpoint} -> {chunk_end_time}")

    is_first_chunk = Path(current_checkpoint) == checkpoint_path

    last_res = run_adaptive_reward_from_snapshot_chronological(
        snapshot_path=str(current_checkpoint),
        end_time=chunk_end_time,

        # Only needed for the first chunk from the clean pre-2020 checkpoint.
        sim_start="2020-01-01 09:30:00" if is_first_chunk else None,
        trade_start="2020-01-01 09:30:00" if is_first_chunk else None,

        output_dir=str(chunk_dir),
        save_snapshot_path=str(chunk_checkpoint),

        # First chunk starts a fresh portfolio. Later chunks preserve state from checkpoint.
        reset_execution_state=True if is_first_chunk else False,
        initial_capital=common_kwargs["initial_capital"],
        fee_pct=common_kwargs["fee_pct"],

        dp_lookback_override=common_kwargs["dp_lookback"],
        daily_threshold_lookback_days_override=252,
        threshold_ret_grid_override=five_min_grid,

        autosave_year_start_checkpoints=True,
        verbose=True,
    )

    print(f"[DONE] {year} checkpoint:", chunk_checkpoint)
    current_checkpoint = chunk_checkpoint

print("Final checkpoint:", current_checkpoint)

[TRAIN][DAILY-PROB] n=200 pos=39 (19.50%)
[TRAIN][DAILY-PROB] n=225 pos=45 (20.00%)
[TRAIN][DAILY-PROB] n=250 pos=54 (21.60%)
[TRAIN][DAILY-PROB] n=275 pos=63 (22.91%)
[TRAIN][DAILY-PROB] n=300 pos=70 (23.33%)
[TRAIN][DAILY-PROB] n=325 pos=79 (24.31%)
[TRAIN][DAILY-PROB] n=350 pos=86 (24.57%)
[TRAIN][DAILY-PROB] n=375 pos=92 (24.53%)
[TRAIN][DAILY-PROB] n=400 pos=98 (24.50%)
[TRAIN][DAILY-PROB] n=425 pos=100 (23.53%)
[TRAIN][DAILY-PROB] n=450 pos=106 (23.56%)
[TRAIN][DAILY-PROB] n=475 pos=114 (24.00%)
[TRAIN][DAILY-PROB] n=500 pos=123 (24.60%)
[TRAIN][DAILY-PROB] n=525 pos=131 (24.95%)
[TRAIN][DAILY-PROB] n=550 pos=133 (24.18%)
[TRAIN][DAILY-PROB] n=575 pos=140 (24.35%)
[TRAIN][DAILY-PROB] n=600 pos=144 (24.00%)
[TRAIN][DAILY-PROB] n=625 pos=152 (24.32%)
[TRAIN][DAILY-PROB] n=650 pos=155 (23.85%)
[TRAIN][DAILY-PROB] n=675 pos=161 (23.85%)
[TRAIN][DAILY-PROB] n=700 pos=163 (23.29%)
[TRAIN][DAILY-PROB] n=725 pos=166 (22.90%)
[TRAIN][DAILY-PROB] n=750 pos=174 (23.20%)
[TRAIN][DAILY-PROB] 

,date,decision_date,result_date,p_day,p_day_source_date,daily_gate_mode,daily_reward_mode,direct_gate_confidence,direct_gate_prob_force_buy,direct_gate_prob_free,...,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
1979,2019-12-23,2019-12-23,2019-12-24,0.182384,2019-12-23,threshold,counterfactual_5m,NaN,NaN,NaN,...,FORCE_BUY,-0.000715,-0.000715,-0.000715,-0.000715,FORCE_BUY,3.471777e+06,321.24,0.0,2.0
1980,2019-12-24,2019-12-24,2019-12-26,0.163509,2019-12-24,threshold,counterfactual_5m,NaN,NaN,NaN,...,FORCE_BUY,0.005754,0.000653,0.000653,0.005754,FORCE_BUY,3.491752e+06,323.39,1.0,0.0
1981,2019-12-26,2019-12-26,2019-12-27,0.150144,2019-12-26,threshold,counterfactual_5m,NaN,NaN,NaN,...,FORCE_BUY,-0.003770,-0.000031,-0.000031,-0.003770,FREE,3.491644e+06,322.39,1.0,0.0
1982,2019-12-27,2019-12-27,2019-12-30,0.195312,2019-12-27,threshold,counterfactual_5m,NaN,NaN,NaN,...,FORCE_BUY,-0.005354,0.003275,-0.000093,-0.005354,FREE,3.503080e+06,321.38,1.0,0.0
1983,2019-12-30,2019-12-30,2019-12-31,0.253078,2019-12-30,threshold,counterfactual_5m,NaN,NaN,NaN,...,FORCE_BUY,0.002551,0.002551,0.002551,0.002551,FORCE_BUY,3.512017e+06,322.24,1.5,0.0


,date,equity,cash,pos,qty,entry_px,decision_date,result_date,buy_th,sell_th,...,daily_buy_level,daily_sell_level,decision_extreme_base_dir,decision_extreme_region,decision_extreme_label,decision_extreme_window_end_date,decision_extreme_ref_high,decision_extreme_ref_low,decision_extreme_future_max_high,decision_extreme_future_min_low
1979,2019-12-24,162264.531867,0.0,1,505.119325,310.67,2019-12-23,2019-12-24,1.0,0.0,...,0.29,0.31,sell,sell_high_broken,0.0,2019-12-31,321.65,321.06,323.8,320.15
1980,2019-12-26,163350.538416,0.0,1,505.119325,310.67,2019-12-24,2019-12-26,1.0,0.0,...,0.29,0.31,sell,pending,NaN,NaT,NaN,NaN,NaN,NaN
1981,2019-12-27,162845.419091,0.0,1,505.119325,310.67,2019-12-26,2019-12-27,1.0,0.0,...,0.29,0.31,sell,pending,NaN,NaT,NaN,NaN,NaN,NaN
1982,2019-12-30,162335.248573,0.0,1,505.119325,310.67,2019-12-27,2019-12-30,1.5,0.0,...,0.29,0.31,sell,pending,NaN,NaT,NaN,NaN,NaN,NaN
1983,2019-12-31,162769.651192,0.0,1,505.119325,310.67,2019-12-30,2019-12-31,1.5,0.0,...,0.29,0.31,sell,pending,NaN,NaT,NaN,NaN,NaN,NaN


[RUN] 2020: output_SPY_build_prev_daily_context_at_2020\fresh_checkpoint_prev_daily_context__before_2020.joblib -> 2020-12-31 16:00:00
[CHRONO] 5m day=2019-12-31 rows=358612:358654
[TRAIN][5M] asof=2019-12-31 feats=114 buy=YES sell=YES rows=65593
[CHECKPOINT] autosaved chronological year-start snapshot for 2020 before 2020-01-02 -> output_SPY_chrono_prev_daily_context_wide_5m_resumable\chunk_2020\year_start_checkpoints\continued_checkpoint__through_2020__start_2020.joblib
[CHRONO] 5m day=2020-01-02 rows=358654:358842
[CHRONO] daily close=2020-01-02 idx=2489
[CHRONO] 5m day=2020-01-03 rows=358842:359032
[CHRONO] daily close=2020-01-03 idx=2490
[CHRONO] 5m day=2020-01-06 rows=359032:359222
[TRAIN][5M] asof=2020-01-06 feats=114 buy=YES sell=YES rows=65719
[CHRONO] daily close=2020-01-06 idx=2491
[CHRONO] 5m day=2020-01-07 rows=359222:359409
[CHRONO] daily close=2020-01-07 idx=2492
[CHRONO] 5m day=2020-01-08 rows=359409:359602
[CHRONO] daily close=2020-01-08 idx=2493
[CHRONO] 5m day=2020-0

c:\Users\TonyTang\Documents\chan.py\adaptive_reward_checkpoint_fresh.py:3132: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  daily_log_df = pd.concat(daily_log_frames, ignore_index=True, sort=False) if daily_log_frames else pd.DataFrame()


...

[SAVED] output_SPY_chrono_prev_daily_context_wide_5m_resumable\chunk_2024\trades.csv
[SAVED] output_SPY_chrono_prev_daily_context_wide_5m_resumable\chunk_2024\signal_decisions.csv
[SAVED] output_SPY_chrono_prev_daily_context_wide_5m_resumable\chunk_2024\daily_log.csv
[SAVED] output_SPY_chrono_prev_daily_context_wide_5m_resumable\chunk_2024\daily_reward_log.csv
[CHECKPOINT] saved chronological snapshot -> output_SPY_chrono_prev_daily_context_wide_5m_resumable\chunk_2024\continued_checkpoint__through_2024.joblib
[DONE] 2024 checkpoint: output_SPY_chrono_prev_daily_context_wide_5m_resumable\chunk_2024\continued_checkpoint__through_2024.joblib
[RUN] 2025: output_SPY_chrono_prev_daily_context_wide_5m_resumable\chunk_2024\continued_checkpoint__through_2024.joblib -> 2025-12-31 16:00:00
[CHRONO] 5m day=2024-12-31 rows=596267:596310
[TRAIN][5M] asof=2024-12-31 feats=114 buy=YES sell=YES rows=113566
[CHECKPOINT] autosaved chronological year-start snapshot for 2025 before 2025-01-02 -> output_SP

c:\Users\TonyTang\Documents\chan.py\adaptive_reward_checkpoint_fresh.py:3132: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  daily_log_df = pd.concat(daily_log_frames, ignore_index=True, sort=False) if daily_log_frames else pd.DataFrame()


[SAVED] output_SPY_chrono_prev_daily_context_wide_5m_resumable\chunk_2025\trades.csv
[SAVED] output_SPY_chrono_prev_daily_context_wide_5m_resumable\chunk_2025\signal_decisions.csv
[SAVED] output_SPY_chrono_prev_daily_context_wide_5m_resumable\chunk_2025\daily_log.csv
[SAVED] output_SPY_chrono_prev_daily_context_wide_5m_resumable\chunk_2025\daily_reward_log.csv
[CHECKPOINT] saved chronological snapshot -> output_SPY_chrono_prev_daily_context_wide_5m_resumable\chunk_2025\continued_checkpoint__through_2025.joblib
[DONE] 2025 checkpoint: output_SPY_chrono_prev_daily_context_wide_5m_resumable\chunk_2025\continued_checkpoint__through_2025.joblib
[RUN] 2026: output_SPY_chrono_prev_daily_context_wide_5m_resumable\chunk_2025\continued_checkpoint__through_2025.joblib -> 2026-06-12 16:00:00
[CHRONO] 5m day=2025-12-31 rows=643744:643791
[TRAIN][5M] asof=2025-12-31 feats=114 buy=YES sell=YES rows=122755
[CHECKPOINT] autosaved chronological year-start snapshot for 2026 before 2026-01-02 -> output_SP

c:\Users\TonyTang\Documents\chan.py\adaptive_reward_checkpoint_fresh.py:3132: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  daily_log_df = pd.concat(daily_log_frames, ignore_index=True, sort=False) if daily_log_frames else pd.DataFrame()


[SAVED] output_SPY_chrono_prev_daily_context_wide_5m_resumable\chunk_2026\trades.csv
[SAVED] output_SPY_chrono_prev_daily_context_wide_5m_resumable\chunk_2026\signal_decisions.csv
[SAVED] output_SPY_chrono_prev_daily_context_wide_5m_resumable\chunk_2026\daily_log.csv
[SAVED] output_SPY_chrono_prev_daily_context_wide_5m_resumable\chunk_2026\daily_reward_log.csv
[CHECKPOINT] saved chronological snapshot -> output_SPY_chrono_prev_daily_context_wide_5m_resumable\chunk_2026\continued_checkpoint__through_2026.joblib
[DONE] 2026 checkpoint: output_SPY_chrono_prev_daily_context_wide_5m_resumable\chunk_2026\continued_checkpoint__through_2026.joblib
Final checkpoint: output_SPY_chrono_prev_daily_context_wide_5m_resumable\chunk_2026\continued_checkpoint__through_2026.joblib
